<a href="https://colab.research.google.com/github/ellianorsX/computer_vision/blob/main/menu_detector_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [5]:
print('Menu detector!')

Menu detector!


In [24]:
# =========================
# Import Libraries
# =========================

from google.colab import drive
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision.transforms as transforms
from torchvision.models import mobilenet_v2


from PIL import Image, UnidentifiedImageError
from torch.utils.data import Dataset, DataLoader
import os
import numpy as np

In [7]:
drive.mount('/content/drive')


Mounted at /content/drive


In [17]:
# Define Dataset Path

DATASET_PATH = '/content/drive/MyDrive/food101_dataset/'
print('Dataset_path:', DATASET_PATH)




CUSTOM_CLASS_MAPPING = {
    "hamburger": "hamburger",
    "pizza": "pizza",
    "kebab": "kebab",
    "pilaf": "pilaf",
}

CLASSES = ["hamburger", "pizza",  "kebab", "pilaf"]
CLASS_TO_IDX = {cls: i for i, cls in enumerate(CLASSES)}
NUM_CLASSES = len(CLASSES)


print(NUM_CLASSES)
print(CLASS_TO_IDX)

# Alohida train va val transformlarini yaratamiz
train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(15),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

val_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])


Dataset_path: /content/drive/MyDrive/food101_dataset/
4
{'hamburger': 0, 'pizza': 1, 'kebab': 2, 'pilaf': 3}


In [9]:
# =========================
# Custom Dataset Class
# =========================

class FoodDataset(Dataset):
    def __init__(self, images, labels, transform=None):
        self.images = images
        self.labels = labels
        self.transform = transform

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        img_path = self.images[idx]
        label = self.labels[idx]

        try:
            image = Image.open(img_path).convert("RGB")
        except (UnidentifiedImageError, OSError):
            print(f"Skipping broken image: {img_path}")
            return self.__getitem__((idx + 1) % len(self.images))

        if self.transform:
            image = self.transform(image)

        return image, label

In [14]:
# =========================
# Gather and Split Data
# =========================

all_images = []

for original_class, mapped_class in CUSTOM_CLASS_MAPPING.items():
    class_path = os.path.join(DATASET_PATH, original_class)
    print("class_path:", class_path)

    if not os.path.exists(class_path):
        print(f"Warning: {class_path} not found")
        continue

    for img in os.listdir(class_path):
        if img.endswith((".jpg", ".jpeg", ".png")):
            full_path = os.path.join(class_path, img)
            all_images.append(
                (full_path, CLASS_TO_IDX[mapped_class])
            )

np.random.shuffle(all_images)

split = int(0.8 * len(all_images))
train_data = all_images[:split]
val_data = all_images[split:]

train_images, train_labels = zip(*train_data)
val_images, val_labels = zip(*val_data)

# print("all_images:", all_images)

dataset = FoodDataset(train_images, train_labels)
print(len(dataset))

img, lbl = dataset[0]

class_path: /content/drive/MyDrive/food101_dataset/hamburger
class_path: /content/drive/MyDrive/food101_dataset/pizza
class_path: /content/drive/MyDrive/food101_dataset/kebab
class_path: /content/drive/MyDrive/food101_dataset/pilaf
1639


In [18]:
train_dataset = FoodDataset(
    train_images,
    train_labels,
    transform=train_transform,
)

val_dataset = FoodDataset(
    val_images,
    val_labels,
    transform=val_transform,
)

In [20]:
train_loader = DataLoader(
    train_dataset,
    batch_size=32,
    shuffle=True,
    num_workers=2,  # thread #paraller loading for speed
)

val_loader = DataLoader(
    val_dataset,
    batch_size=32,
    shuffle=False,
    num_workers=2,
)

In [22]:
# Pretrained model
model = mobilenet_v2(weights="IMAGENET1K_V1")  # pretrained model | lightweight | CNN | 1000 classes | 1 million
model.classifier[1] = nn.Linear(
    model.classifier[1].in_features,
    NUM_CLASSES,
)  # fine-tuning | backbone | model layer freeze

Downloading: "https://download.pytorch.org/models/mobilenet_v2-b0353104.pth" to /root/.cache/torch/hub/checkpoints/mobilenet_v2-b0353104.pth


100%|██████████| 13.6M/13.6M [00:00<00:00, 127MB/s]


In [23]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device", device)

model = model.to(device)

device cuda


In [30]:
criterion = nn.CrossEntropyLoss() # Loss Function | '70%' burger, '30%' pilaf
optimizer = optim.Adam(model.parameters(), lr=0.001) # wight
torch.backends.cudnn.benchmark = True # Benchmark Setting | Trick | 10-20% speed


